# 🔎 Análise Exploratória de Dados — Detecção de Fraudes

Este notebook analisa o dataset de transações de cartão de crédito utilizado no projeto.

## 1. Importação das bibliotecas

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
print('Bibliotecas carregadas com sucesso.')

## 2. Carregamento dos dados

In [ ]:
DATASET_PATH = os.path.join('..', 'data', 'raw', 'creditcard.csv')
DATASET_URL = 'https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv'
os.makedirs(os.path.dirname(DATASET_PATH), exist_ok=True)

if not os.path.exists(DATASET_PATH):
    print('Dataset não encontrado localmente. Baixando...')
    df = pd.read_csv(DATASET_URL)
    df.to_csv(DATASET_PATH, index=False)
    print('Download concluído!')
else:
    print('Dataset encontrado localmente.')
    df = pd.read_csv(DATASET_PATH)

print(f'Dimensão: {df.shape}')

## 3. Visão geral

In [ ]:
print('Linhas:', df.shape[0])
print('Colunas:', df.shape[1])
display(df.head())
display(df.dtypes.to_frame('dtype'))

## 4. Qualidade dos dados

In [ ]:
missing = df.isnull().sum()
quality = pd.DataFrame({'missing': missing, 'missing_percent': missing / len(df) * 100})
display(quality[quality['missing'] > 0])
print('Valores ausentes:', int(missing.sum()))
print('Registros duplicados:', int(df.duplicated().sum()))

## 5. Estatística descritiva

In [ ]:
display(df.describe().T)

## 6. Distribuição da variável alvo

`Class = 0` representa transações legítimas e `Class = 1` representa fraudes.

In [ ]:
class_counts = df['Class'].value_counts().sort_index()
class_percent = df['Class'].value_counts(normalize=True).sort_index() * 100
summary = pd.DataFrame({'quantidade': class_counts, 'percentual': class_percent})
display(summary)

In [ ]:
plt.figure(figsize=(8,5))
bars = plt.bar(['Legítimas (0)', 'Fraudes (1)'], class_counts.values)
plt.title('Distribuição das Transações por Classe')
plt.ylabel('Quantidade')
for bar, value in zip(bars, class_counts.values):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height(), f'{value:,}', ha='center', va='bottom')
plt.tight_layout()
plt.show()

### Insight

O forte desbalanceamento torna a Accuracy insuficiente como métrica isolada. Precision, Recall, F1-Score, ROC-AUC e Average Precision são mais adequadas.

## 7. Análise de `Amount`

In [ ]:
display(df['Amount'].describe().to_frame('Amount'))
plt.figure(figsize=(10,5))
plt.hist(df['Amount'], bins=100)
plt.title('Distribuição dos Valores das Transações')
plt.xlabel('Amount')
plt.ylabel('Frequência')
plt.tight_layout()
plt.show()

### Transformação logarítmica

`Amount_log` é criada no pré-processamento. Atualmente os modelos utilizam `Amount` após normalização com `StandardScaler`.

In [ ]:
df['Amount_log'] = np.log1p(df['Amount'])
plt.figure(figsize=(10,5))
plt.hist(df['Amount_log'], bins=100)
plt.title('Distribuição de Amount após log1p')
plt.xlabel('log1p(Amount)')
plt.ylabel('Frequência')
plt.tight_layout()
plt.show()

In [ ]:
display(df.groupby('Class')['Amount'].agg(['count','mean','median','std','min','max']))

In [ ]:
plt.figure(figsize=(9,6))
plt.boxplot([df.loc[df['Class']==0,'Amount'], df.loc[df['Class']==1,'Amount']], labels=['Legítimas','Fraudes'], showfliers=False)
plt.title('Distribuição de Amount por Classe')
plt.ylabel('Amount')
plt.tight_layout()
plt.show()

## 8. Análise temporal

In [ ]:
print(f"Time mínimo: {df['Time'].min():,.2f}")
print(f"Time máximo: {df['Time'].max():,.2f}")
plt.figure(figsize=(12,5))
plt.hist(df['Time'], bins=100)
plt.title('Distribuição Temporal das Transações')
plt.xlabel('Time')
plt.ylabel('Quantidade')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12,5))
plt.hist(df.loc[df['Class']==0,'Time'], bins=100, alpha=0.7, label='Legítimas')
plt.hist(df.loc[df['Class']==1,'Time'], bins=100, alpha=0.7, label='Fraudes')
plt.title('Distribuição Temporal por Classe')
plt.xlabel('Time')
plt.ylabel('Quantidade')
plt.legend()
plt.tight_layout()
plt.show()

Para produção, uma evolução importante seria utilizar validação temporal, evitando que informações futuras influenciem o treinamento.

## 9. Correlação com `Class`

In [ ]:
correlations = (df.drop(columns=['Amount_log']).corr(numeric_only=True)['Class'].drop('Class').sort_values(key=lambda x: x.abs(), ascending=False))
display(correlations.to_frame('correlacao_com_Class').head(15))

In [ ]:
top_features = correlations.head(10)
plt.figure(figsize=(10,6))
plt.barh(top_features.index[::-1], top_features.values[::-1])
plt.title('Top 10 Variáveis por Correlação com Class')
plt.xlabel('Correlação')
plt.ylabel('Variável')
plt.tight_layout()
plt.show()

## 10. Distribuição das principais variáveis

Correlação é apenas uma análise inicial; modelos podem capturar relações não lineares.

In [ ]:
for feature in top_features.head(4).index:
    plt.figure(figsize=(10,5))
    plt.hist(df.loc[df['Class']==0, feature], bins=80, alpha=0.7, label='Legítimas')
    plt.hist(df.loc[df['Class']==1, feature], bins=80, alpha=0.7, label='Fraudes')
    plt.title(f'Distribuição de {feature} por Classe')
    plt.xlabel(feature)
    plt.ylabel('Frequência')
    plt.legend()
    plt.tight_layout()
    plt.show()

## 11. Principais descobertas

- Forte desbalanceamento entre as classes.
- `Amount` apresenta distribuição assimétrica.
- Algumas variáveis PCA possuem maior associação com `Class`.
- Correlação isolada não é suficiente para identificar fraudes.
- Métricas adequadas à classe minoritária são essenciais.
- O threshold altera o equilíbrio entre Precision e Recall.
- Validação temporal é uma evolução relevante para produção.

## 12. Próximos passos

O pipeline segue para pré-processamento, divisão treino/teste, StandardScaler, SMOTE nos modelos de árvore, treinamento de Logistic Regression, Random Forest e XGBoost, avaliação, análise de threshold e SHAP.

### Evoluções futuras
- Avaliar o uso efetivo de `Amount_log`.
- Implementar validação temporal.
- Realizar tuning de hiperparâmetros.
- Avaliar custos financeiros de falsos positivos e falsos negativos.

---
# Conclusão

A análise exploratória demonstra que a detecção de fraude é um problema altamente desbalanceado e exige uma abordagem orientada à classe minoritária.

**Projeto:** Detecção de Fraudes em Cartões de Crédito  
**Autor:** Elton Jhon Silva